# Análise Final de Desempenho em Lógica Proposicional

Este notebook consolida e analisa os resultados da avaliação de diferentes modelos e metodologias no nosso dataset de Lógica Proposicional, replicando as métricas do artigo LogicBench.

### Métricas do Artigo LogicBench

O artigo original utiliza as seguintes métricas para avaliar o desempenho dos modelos:

**1. Para a tarefa BQA (Binary Question Answering):**
- **Acurácia Geral:** A porcentagem total de respostas corretas.
  - `Fórmula: (Total de Acertos) / (Total de Perguntas)`
- **A(Yes):** A acurácia calculada apenas sobre o subconjunto de perguntas cuja resposta correta é "Sim".
  - `Fórmula: (Total de Acertos 'Sim') / (Total de Perguntas 'Sim')`
- **A(No):** A acurácia calculada apenas sobre o subconjunto de perguntas cuja resposta correta é "Não".
  - `Fórmula: (Total de Acertos 'Não') / (Total de Perguntas 'Não')`

**2. Para a tarefa MCQA (Multiple Choice Question Answering):**
- **Acurácia Geral:** A porcentagem de vezes que o modelo escolheu a opção correta.
  - `Fórmula: (Total de Acertos) / (Total de Perguntas)`

**3. Para a tarefa de Tradução para Z3:**
- **Acurácia de Tradução:** A porcentagem de vezes que a fórmula gerada pelo LLM, quando avaliada pelo Z3, resulta em uma consequência lógica válida.
  - `Fórmula: (Total de Traduções Corretas) / (Total de Tarefas)`

Este notebook irá calcular e apresentar todas essas métricas para os modelos disponíveis na planilha de resultados.

In [2]:
import pandas as pd
import json
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# --- CONFIGURAÇÕES ---
# Define o caminho raiz do projeto (LLM-Logic-Eval) de forma dinâmica
PROJECT_ROOT = Path.cwd().parent

# Caminhos para os arquivos de dados
EVALUATION_FILE_PATH = PROJECT_ROOT / "model_evaluation" / "dataframes" / "evaluation_spreadsheet.xlsx"
Z3_RESULTS_PATH = PROJECT_ROOT / "model_evaluation" / "z3_evaluation"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Lendo resultados da avaliação de: {EVALUATION_FILE_PATH}")
print(f"Lendo resultados do Z3 de: {Z3_RESULTS_PATH}")

Raiz do projeto: d:\IFCE\2025.2_LOGICOMP\LLM-Logic-Eval
Lendo resultados da avaliação de: d:\IFCE\2025.2_LOGICOMP\LLM-Logic-Eval\model_evaluation\dataframes\evaluation_spreadsheet.xlsx
Lendo resultados do Z3 de: d:\IFCE\2025.2_LOGICOMP\LLM-Logic-Eval\model_evaluation\z3_evaluation


In [6]:
def robust_normalize_bqa_answer(text):
    """Normaliza a resposta BQA para 'Sim', 'Não' ou 'Inválida'."""
    if not isinstance(text, str) or pd.isna(text): return "Inválida"
    cleaned_text = text.lower().strip()
    if cleaned_text.startswith('sim'): return 'Sim'
    if cleaned_text.startswith('não'): return 'Não'
    return "Inválida"

def normalize_mcqa_answer(text):
    """Normaliza a resposta MCQA para um índice numérico (0-3) ou -1 se inválido."""
    if pd.isna(text): return -1
    text_str = str(text)
    match = re.search(r'^\s*([0-3])\s*$', text_str)
    if match: return int(match.group(1))
    return -1

def load_and_process_results(excel_path):
    """Carrega a planilha, normaliza as respostas e calcula a correção para cada modelo."""
    try:
        df = pd.read_excel(excel_path, sheet_name='evaluation_spreadsheet (1)') # Assumindo que a aba principal se chama 'Sheet1'
        print(f"Planilha carregada com {len(df)} linhas.")
    except Exception as e:
        print(f"ERRO ao carregar a planilha: {e}")
        return None

    # Detecta automaticamente quais modelos foram testados
    model_cols = [col for col in df.columns if col.endswith('_answer') and col != 'correct_answer']
    print(f"Modelos detectados na planilha: {[col.replace('_answer', '') for col in model_cols]}")

    # Corrige o tipo da coluna de resposta correta para MCQA
    mcqa_mask = df['task_type'] == 'MCQA'
    df.loc[mcqa_mask, 'correct_answer'] = pd.to_numeric(df.loc[mcqa_mask, 'correct_answer'], errors='coerce')

    for col in model_cols:
        is_correct_col = col + '_is_correct'
        
        # Aplica a normalização e a comparação correta, tratando valores faltantes (NaN)
        df[is_correct_col] = df.apply(
            lambda row: 
                (robust_normalize_bqa_answer(row[col]) == row['correct_answer']) if row['task_type'] == 'BQA' and pd.notna(row[col])
                else (normalize_mcqa_answer(row[col]) == row['correct_answer']) if row['task_type'] == 'MCQA' and pd.notna(row[col])
                else False, # Considera a resposta como incorreta se estiver vazia
            axis=1
        )
    return df

# Executa o carregamento
df_results = load_and_process_results(EVALUATION_FILE_PATH)

Planilha carregada com 350 linhas.
Modelos detectados na planilha: ['gemini_2.5_pro', 'gemini_2.5_flash', 'chat_gpt', 'deep_seek']


In [8]:
if df_results is not None:
    print("\n" + "="*50 + "\nResultados da Avaliação BQA\n" + "="*50)
    
    bqa_df = df_results[df_results['task_type'] == 'BQA'].copy()
    summary_bqa_data = []
    model_cols = [col for col in df_results.columns if col.endswith('_answer') and col != 'correct_answer']
    
    for col in model_cols:
        model_name = col.replace('_answer', '')
        is_correct_col = col + '_is_correct'
        
        # Calcula métricas apenas sobre as linhas que têm resposta para este modelo
        valid_responses_df = bqa_df.dropna(subset=[col])
        
        if not valid_responses_df.empty:
            yes_q = valid_responses_df[valid_responses_df['correct_answer'] == 'Sim']
            a_yes = yes_q[is_correct_col].mean() * 100 if not yes_q.empty else 0
            
            no_q = valid_responses_df[valid_responses_df['correct_answer'] == 'Não']
            a_no = no_q[is_correct_col].mean() * 100 if not no_q.empty else 0
            
            overall = valid_responses_df[is_correct_col].mean() * 100
            
            summary_bqa_data.append({
                "Modelo": model_name,
                "Acurácia Geral": f"{overall:.2f}%",
                "A(Yes)": f"{a_yes:.2f}%",
                "A(No)": f"{a_no:.2f}%",
                "Respostas Válidas": f"{len(valid_responses_df)}/{len(bqa_df)}"
            })
        else:
            summary_bqa_data.append({"Modelo": model_name, "Acurácia Geral": "N/A", "A(Yes)": "N/A", "A(No)": "N/A", "Respostas Válidas": f"0/{len(bqa_df)}"})

    summary_bqa_df = pd.DataFrame(summary_bqa_data).set_index('Modelo')
    print("\nTabela 1 (BQA): Desempenho Geral")
    display(summary_bqa_df)



Resultados da Avaliação BQA

Tabela 1 (BQA): Desempenho Geral


,Acurácia Geral,A(Yes),A(No),Respostas Válidas
Modelo,,,,
gemini_2.5_pro,94.90%,96.88%,93.94%,196/260
gemini_2.5_flash,90.82%,96.88%,87.88%,196/260
chat_gpt,N/A,N/A,N/A,0/260
deep_seek,N/A,N/A,N/A,0/260


In [12]:
if df_results is not None:
    print("\n" + "="*50 + "\nResultados da Avaliação MCQA\n" + "="*50)
    
    mcqa_df = df_results[df_results['task_type'] == 'MCQA'].copy()
    summary_mcqa_data = []
    model_cols = [col for col in df_results.columns if col.endswith('_answer') and col != 'correct_answer']

    for col in model_cols:
        model_name = col.replace('_answer', '')
        is_correct_col = col + '_is_correct'
        
        valid_responses_df = mcqa_df.dropna(subset=[col])
        
        if not valid_responses_df.empty:
            accuracy = valid_responses_df[is_correct_col].mean() * 100
            summary_mcqa_data.append({"Modelo": model_name, "Acurácia Geral": f"{accuracy:.2f}%", "Respostas Válidas": f"{len(valid_responses_df)}/{len(mcqa_df)}"})
        else:
            summary_mcqa_data.append({"Modelo": model_name, "Acurácia Geral": "N/A", "Respostas Válidas": f"0/{len(mcqa_df)}"})

    summary_mcqa_df = pd.DataFrame(summary_mcqa_data).set_index('Modelo')
    print("\nTabela 2 (MCQA): Desempenho Geral")
    display(summary_mcqa_df)

    # Tabela de acurácia por regra
    accuracy_mcqa_by_rule = mcqa_df.groupby('rule').apply(
        lambda x: pd.Series({
            model.replace('_answer', ''): f"{x[model + '_is_correct'].mean() * 100:.2f}%"
            for model in model_cols if not x.dropna(subset=[model]).empty
        })
    )
    print("\nTabela 3 (MCQA): Acurácia por Regra de Inferência")
    display(accuracy_mcqa_by_rule)


Resultados da Avaliação MCQA

Tabela 2 (MCQA): Desempenho Geral


,Acurácia Geral,Respostas Válidas
Modelo,,
gemini_2.5_pro,83.33%,90/90
gemini_2.5_flash,54.44%,90/90
chat_gpt,N/A,0/90
deep_seek,N/A,0/90



Tabela 3 (MCQA): Acurácia por Regra de Inferência


C:\Users\Iuri\AppData\Local\Temp\ipykernel_10132\2667191967.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_mcqa_by_rule = mcqa_df.groupby('rule').apply(


,gemini_2.5_pro,gemini_2.5_flash
rule,,
bidirectional_dilemma,90.00%,20.00%
commutation,40.00%,10.00%
constructive_dilemma,100.00%,80.00%
destructive_dilemma,100.00%,10.00%
disjunctive_syllogism,90.00%,100.00%
hypothetical_syllogism,100.00%,90.00%
material_implication,40.00%,20.00%
modus_ponens,100.00%,100.00%
modus_tollens,90.00%,60.00%


In [11]:

# --- Análise dos Resultados do Z3 ---

print("\n" + "="*50)
print("Resultados da Avaliação Z3 (Tradução LLM -> Z3)")
print("="*50)

def check_z3_conclusion(rule_name, model_dict):
    """
    Verifica se o modelo (dicionário de valoração) do Z3 satisfaz a conclusão lógica da regra.
    Retorna True se a conclusão for válida, False caso contrário.
    """
    if not model_dict:
        return False
    
    # Converte os valores do modelo de string para booleano (ex: "True" -> True)
    m = {k: (v == 'True') for k, v in model_dict.items()}
    
    try:
        # Para cada regra, definimos a fórmula da conclusão que esperamos que seja verdadeira
        if rule_name == 'modus_ponens': return m['q']
        if rule_name == 'modus_tollens': return not m['p']
        if rule_name == 'hypothetical_syllogism': return not m['p'] or m['r'] # Equivalente a p -> r
        if rule_name == 'disjunctive_syllogism': return m['q']
        if rule_name == 'constructive_dilemma': return m['q'] or m['s']
        if rule_name == 'destructive_dilemma': return not m['p'] or not m['r']
        if rule_name == 'bidirectional_dilemma': return m['q'] or not m['r']
        if rule_name == 'commutation': return m['q'] or m['p']
        if rule_name == 'material_implication': return not m['p'] or m['q']
        # Se a regra não for encontrada, não podemos validar
        return False
    except KeyError:
        # Se uma variável necessária (p, q, etc.) não estiver no modelo, a conclusão não pode ser verificada.
        return False

def load_and_evaluate_z3_results(excel_path):
    """
    Carrega os resultados da aba Z3, avalia a correção de cada tradução e retorna um DataFrame.
    """
    try:
        df_z3 = pd.read_excel(excel_path, sheet_name='Z3_Evaluation')
        print(f"Aba 'Z3_Evaluation' carregada com {len(df_z3)} linhas.")
    except Exception as e:
        print(f"ERRO: Não foi possível ler a aba 'Z3_Evaluation'. Verifique se ela existe no arquivo '{excel_path.name}'. Erro: {e}")
        return None

    # Detecta automaticamente os modelos avaliados
    model_formalization_cols = [col for col in df_z3.columns if col.endswith('_formalization')]
    
    for col in model_formalization_cols:
        model_name = col.replace('_formalization', '')
        result_col = f"{model_name}_z3_result"
        is_correct_col = f"{model_name}_is_correct"
        
        # Avalia se a tradução do LLM foi correta
        # A tradução é correta se o resultado do Z3 (da negação da conclusão) for 'unsat'
        df_z3[is_correct_col] = (df_z3[result_col] == 'unsat')

    return df_z3

# Executa o carregamento e a avaliação
df_z3_evaluated = load_and_evaluate_z3_results(EVALUATION_FILE_PATH)

if df_z3_evaluated is not None:
    # --- Tabela de Desempenho Geral ---
    summary_z3_data = []
    model_cols = [col for col in df_z3_evaluated.columns if col.endswith('_is_correct')]

    for col in model_cols:
        model_name = col.replace('_is_correct', '').replace('_', '-')
        accuracy = df_z3_evaluated[col].mean() * 100
        summary_z3_data.append({"Modelo": f"{model_name} (via Z3)", "Acurácia de Tradução": f"{accuracy:.2f}%"})

    summary_z3_df = pd.DataFrame(summary_z3_data).set_index('Modelo')
    print("\nTabela 4 (Z3): Desempenho Geral na Tarefa de Tradução")
    display(summary_z3_df)

    # --- Tabela de Desempenho por Regra ---
    model_is_correct_cols = [col for col in df_z3_evaluated.columns if col.endswith('_is_correct')]
    
    # Agrupa por regra e calcula a média de acertos para cada modelo
    z3_accuracy_by_rule = df_z3_evaluated.groupby('rule')[model_is_correct_cols].mean()
    
    # Renomeia as colunas para ficarem mais legíveis
    z3_accuracy_by_rule.columns = [col.replace('_is_correct', '').replace('_', '-') for col in z3_accuracy_by_rule.columns]
    
    # Formata como porcentagem
    z3_accuracy_by_rule = z3_accuracy_by_rule.map(lambda x: f"{x*100:.2f}%")
    
    print("\nTabela 5 (Z3): Acurácia de Tradução por Regra")
    display(z3_accuracy_by_rule)


Resultados da Avaliação Z3 (Tradução LLM -> Z3)
Aba 'Z3_Evaluation' carregada com 90 linhas.

Tabela 4 (Z3): Desempenho Geral na Tarefa de Tradução


,Acurácia de Tradução
Modelo,
gemini-2.5-pro (via Z3),97.78%
gemini-2.5-flash (via Z3),100.00%
chat-gpt (via Z3),0.00%
deep-seek (via Z3),0.00%



Tabela 5 (Z3): Acurácia de Tradução por Regra


,gemini-2.5-pro,gemini-2.5-flash,chat-gpt,deep-seek
rule,,,,
Bidirectional_Dilemma,90.00%,100.00%,0.00%,0.00%
Commutation,100.00%,100.00%,0.00%,0.00%
Constructive_Dilemma,100.00%,100.00%,0.00%,0.00%
Destructive_Dilemma,90.00%,100.00%,0.00%,0.00%
Disjunctive_Syllogism,100.00%,100.00%,0.00%,0.00%
Hypothetical_Syllogism,100.00%,100.00%,0.00%,0.00%
Material_Implication,100.00%,100.00%,0.00%,0.00%
Modus_Ponens,100.00%,100.00%,0.00%,0.00%
Modus_Tollens,100.00%,100.00%,0.00%,0.00%
